In [1]:
#install required libs
!pip install transformers datasets torch

  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/10.6 MB 8.0 M


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install "accelerate>=0.26.0"


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#Restart the kernel once the libraries are installed

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from datasets import load_dataset, Dataset
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.model_selection import train_test_split
import torch

In [2]:
!pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 9.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 13.7 MB/s eta 0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]




[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
def tokenize_function(examples):
    # Here we ensure padding and truncation are applied
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "recall": recall_score(labels, predictions, average='macro'),
        "precision": precision_score(labels, predictions, average='macro'),
        "f1": f1_score(labels, predictions, average='macro')
    }

def predict_emotion(model, tokenizer, text):
    # Encoding text to suitable input format for the model
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    # Model Inference
    with torch.no_grad():  # not to keep track of gradients
        outputs = model(**inputs)

    # Probabilities (apply softmax to logits)
    logits = outputs.logits
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    probs = probabilities.detach().numpy()

    # Corresponding labels (assuming model was trained with labels in order)
    labels = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
    return {label: prob for label, prob in zip(labels, probs[0])}

In [6]:
# hugging face dataset on which the training dataset is based
dataset = load_dataset('emotion')

# Load datasets from local CSV files
train_dataset = load_dataset('csv', data_files='hometask_dataset.csv', split='train')
val_dataset = dataset['train'].select(range(8000, 9000))

-----
work-part

In [7]:
# === DQ  ===

train_df = train_dataset.to_pandas()

emotion_to_label = dataset['train'].features['label']._str2int
print(emotion_to_label)

{'sadness': 0, 'joy': 1, 'love': 2, 'anger': 3, 'fear': 4, 'surprise': 5}


In [8]:
print("Label dist:\n", train_df['label'].value_counts())

Label dist:
 label
1.0    256
0.0    187
3.0    133
4.0    101
2.0     97
5.0     61
Name: count, dtype: int64


In [9]:
print("Missing lb:", train_df['label'].isna().sum())

Missing lb: 165


In [10]:
print("Unique emotions:", train_df['emotion'].unique())

Unique emotions: <ArrowStringArray>
['sadness', 'anger', 'love', 'surprise', 'fear', 'joy']
Length: 6, dtype: str


In [11]:
print("Missing emotions:", train_df['emotion'].isna().sum())

Missing emotions: 0


In [12]:
train_df['emotion'] = train_df['emotion'].str.strip()

In [13]:
train_df['label'] = train_df['emotion'].map(emotion_to_label)
train_df['emotion']

0      sadness
1      sadness
2        anger
3         love
4        anger
        ...   
995        joy
996    sadness
997      anger
998    sadness
999        joy
Name: emotion, Length: 1000, dtype: str

In [14]:
train_df = train_df.dropna(subset=['label'])
train_df['label'] = train_df['label'].astype(int)

In [15]:
print("Label distribution:\n", train_df['label'].value_counts())
print("Remaining rows:", len(train_df))

Label distribution:
 label
1    348
0    254
3    150
4    110
2     95
5     43
Name: count, dtype: int64
Remaining rows: 1000


In [16]:
label_names = {v: k for k, v in emotion_to_label.items()}
dist = train_df['label'].value_counts().sort_index()

for label_id, count in dist.items():
    emotion_name = label_names[label_id]
    pct = count / len(train_df) * 100
    bar = '█' * int(pct)
    print(f"{emotion_name:10s} (label {label_id}): {count:4d} samples ({pct:.1f}%) {bar}")

print(f"\nMax class: {dist.max()} | Min class: {dist.min()} | Imbalance ratio: {dist.max()/dist.min():.1f}x")

sadness    (label 0):  254 samples (25.4%) █████████████████████████
joy        (label 1):  348 samples (34.8%) ██████████████████████████████████
love       (label 2):   95 samples (9.5%) █████████
anger      (label 3):  150 samples (15.0%) ███████████████
fear       (label 4):  110 samples (11.0%) ███████████
surprise   (label 5):   43 samples (4.3%) ████

Max class: 348 | Min class: 43 | Imbalance ratio: 8.1x


In [17]:
# uneven distribution,gonna trim exed value ~200 and load underperforming labels from HF

# === DATA BALANCE FIX ===
TARGET = 200
hf_df = dataset['train'].to_pandas()

In [18]:
balanced_parts = []
for label_id in range(6):
    emotion_name = label_names[label_id]
    class_df = train_df[train_df['label'] == label_id]
    current_count = len(class_df)
    if current_count >= TARGET:
        class_df = class_df.sample(TARGET, random_state=42)
    else:
        # from HuggingFace dataset
        needed = TARGET - current_count
        hf_class = hf_df[hf_df['label'] == label_id][['text', 'label']].sample(needed, random_state=42)
        hf_class['emotion'] = emotion_name
        class_df = pd.concat([class_df, hf_class], ignore_index=True)

    balanced_parts.append(class_df)

In [19]:
train_df = pd.concat(balanced_parts, ignore_index=True).sample(frac=1, random_state=42)# shuffle

print(train_df['label'].value_counts().sort_index())
print(f"Total rows: {len(train_df)}")

label
0    200
1    200
2    200
3    200
4    200
5    200
Name: count, dtype: int64
Total rows: 1200


In [ ]:
# # !!!!optional code which can be deleted once data issues are resolved!!!!!
# train_df = train_dataset.to_pandas()
# train_df['label'] = pd.to_numeric(train_df['label'], errors='coerce').fillna(0).astype(int)
# train_dataset = Dataset.from_pandas(train_df)

In [20]:
# === TEXT NORMALIZATION FIX ===
import re

def normalize_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove HTML artifacts like &amp; &hellip; &nbsp;
    text = re.sub(r'&[a-z]+;', ' ', text)
    # 3. Remove special characters, keep only letters, numbers, spaces and basic punctuation
    text = re.sub(r'[^a-z0-9\s\'\.,!?]', ' ', text)
    # 4. Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['text'] = train_df['text'].apply(normalize_text)

print(train_df['text'].head(5).to_string())

1178    i really want to go buy some yardage of art ga...
865     i feel uncomfortable using the word awesome bu...
101     i tend to stop breathing when i m feeling stre...
439            i feel blessed amazed and yes very excited
58      im feeling gloomy as i have completed nothing ...


-----

In [21]:
# Convert back to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)

In [22]:
model_checkpoint = "google/mobilebert-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=6)

Loading weights:   0%|          | 0/1111 [00:00<?, ?it/s]

[transformers] MobileBertForSequenceClassification LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.dense.weight               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architect

In [23]:
# Map function updated with tensor format output (applying tokenization directly on datasets)
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

In [24]:
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2, 
    num_train_epochs=5, 
    weight_decay=0.05,
    fp16=True,
    logging_dir='./logs',
    learning_rate=5e-5
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [27]:
# Train model
trainer.train()

C:\Users\KarynaOhol1\PycharmProjects\GenAI_for_DQE\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Recall,Precision,F1
1,No log,1.793015,0.326000,0.221200,0.128150,0.138892
2,No log,0.903979,0.659000,0.644630,0.572858,0.588690
3,No log,0.659045,0.776000,0.747501,0.726205,0.727239
4,No log,0.781916,0.735000,0.784275,0.672241,0.702612
5,No log,0.760007,0.750000,0.793820,0.693479,0.720917


C:\Users\KarynaOhol1\PycharmProjects\GenAI_for_DQE\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\KarynaOhol1\PycharmProjects\GenAI_for_DQE\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\KarynaOhol1\PycharmProjects\GenAI_for_DQE\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\KarynaOhol1\PycharmProjects\GenAI_for_DQE\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\KarynaOhol1\PycharmProjects\GenAI_for_DQE\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=375, training_loss=169206.34666666668, metrics={'train_runtime': 1136.8468, 'train_samples_per_second': 5.278, 'train_steps_per_second': 0.33, 'total_flos': 94072237056000.0, 'train_loss': 169206.34666666668, 'epoch': 5.0})

In [28]:
# Evaluate the model
results = trainer.evaluate()
print(results)

C:\Users\KarynaOhol1\PycharmProjects\GenAI_for_DQE\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Recall,Precision,F1
No log,0.760007,5,0.750000,0.793820,0.693479,0.720917


{'eval_loss': 0.7600070238113403, 'eval_accuracy': 0.75, 'eval_recall': 0.7938198223497278, 'eval_precision': 0.6934788651956171, 'eval_f1': 0.7209169455084994}


1. 57% accuracy and F1=0.397  - **baseline with dirty data** --is quite poor for a 6-class emotion model
2. **DQ+balance** fix got 78% accuracy and F1=0.746
3. **DQ+balance+norm**  75% accuracy and F1=0.721 - removed punctuation and chars - overkill

In [29]:
# You can store the trained models in different folders to be able to analyze different versions of them with the manual validation code below.
model_path = "trained_model_3"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('trained_model_3\\tokenizer_config.json', 'trained_model_3\\tokenizer.json')

## Manual validation

In [31]:
test_message = "I'm feeling very happy and excited today!"  # expected: joy
test_message2 = "The news from her were astonishing. Such a great news!" # expected: surprise
test_message3 = "If he does that again, I will break his neck!" # expected: anger

In [32]:
# your fine-tunned model
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path)
loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]

In [33]:
# fine-tunned external model
classifier = pipeline("text-classification",model='bhadresh-savani/distilbert-base-uncased-emotion', return_all_scores=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## Your model predictions

In [34]:
predictions = predict_emotion(loaded_model, loaded_tokenizer, test_message)
print(test_message, predictions)
predictions2 = predict_emotion(loaded_model, loaded_tokenizer, test_message2)
print(test_message2, predictions2)
predictions3 = predict_emotion(loaded_model, loaded_tokenizer, test_message3)
print(test_message3, predictions3)

I'm feeling very happy and excited today! {'sadness': np.float32(2.8799204e-05), 'joy': np.float32(0.9928798), 'love': np.float32(0.006375603), 'anger': np.float32(0.0001436071), 'fear': np.float32(6.5825545e-05), 'surprise': np.float32(0.00050652295)}
The news from her were astonishing. Such a great news! {'sadness': np.float32(1.3600781e-07), 'joy': np.float32(1.7518447e-05), 'love': np.float32(3.5860724e-06), 'anger': np.float32(1.3048019e-06), 'fear': np.float32(0.00061228866), 'surprise': np.float32(0.99936515)}
If he does that again, I will break his neck! {'sadness': np.float32(0.21375662), 'joy': np.float32(0.0153529765), 'love': np.float32(0.02033809), 'anger': np.float32(0.69767296), 'fear': np.float32(0.051461026), 'surprise': np.float32(0.0014183341)}


run 1 as is :
- Confused surprise → joy; anger → sadness;

run DQ+balance:
- Confused anger(36,6%) → sadness(42%);

run DQ+balance+norm:
- All 3 correct with high confidence (anger correctly predicted at 69.8%)

## External model predictions

In [18]:
prediction = classifier(test_message)
prediction2 = classifier(test_message2)
prediction3 = classifier(test_message3)
print(test_message, prediction)
print(test_message2, prediction2)
print(test_message3, prediction3)

I'm feeling very happy and excited today! [{'label': 'joy', 'score': 0.999071478843689}]
The news from her were astonishing. Such a great news! [{'label': 'surprise', 'score': 0.8597108721733093}]
If he does that again, I will break his neck! [{'label': 'anger', 'score': 0.647698163986206}]


external model got all correct with hight confidence (0.64–0.99)

NOTES:
- run 1 as is (dirty data):
  - 57% accuracy and F1=0.397
  - Got only 1/3 test sentences correct (confused surprise→joy, anger→sadness)
- run 2 DQ+Balance:
  - 78.2% accuracy and F1=0.746
  - Got 2/3 test sentences correct (anger still confused with sadness 42% vs anger 36%)
- run 3 DQ+Balance+Normalization:
  - 75.0% accuracy and F1=0.721
  - Got **3/3 test sentences correct** ✅ (anger correctly predicted at 69.8%)
- **Key trade-off discovered:**
  - Run 2 has better aggregate metrics but fails 1 real test
  - Run 3 has slightly lower aggregate metrics but passes ALL real tests
  - Aggregate metrics alone don't tell the full story — always validate with real examples!
- **Key insights:**
  - Data balance had the biggest impact — fixing surprise (43→200 samples) fixed test 2 completely
  - Normalization fixed the hardest case (anger vs sadness) by cleaning noisy mixed-case text